# scratch · v3 claim_id fix — join raw's Claimnumber_CLAIM back onto every v3 file

v3's first export keyed everything on `ID_CLAIM` (the repo's DB row id) instead of the
business claim number. `raw_v3.parquet` is the only file that carries both — it was exported
as-is, so it has `claim_id` (== `ID_CLAIM`, what the export wrongly used as the key) and
`Claimnumber_CLAIM` (the real one) side by side.

**The originals were moved aside by hand**, not touched by this notebook:
```
inputs/v3_original/{raw_v3, features_v3_*, targets_v3_*}.parquet
detection/v3_original/v3_scores_*.parquet
```

`raw_v3.parquet` was pulled as **one separate extraction**, while `features_v3_{train,test,oot}`
came from **three separate Z: files** — so raw's `ID_CLAIM` numbering is not guaranteed to
overlap the splits' at all (confirmed: joining features_v3_train straight to raw left ~all rows
unmatched). The safer bridge, when it exists, is a file's **own** `Claimnumber_CLAIM` — features
keeps every column from its transformed frame, so it often already carries one, correct by
construction. targets/scores/attributions never carry their own (01_export_v3 only ever selected
`ID_CLAIM` for them), but they came from the SAME transformed frame as that split's features file,
so that split's `{ID_CLAIM: Claimnumber_CLAIM}` pairing is guaranteed to match them too.

So the bridge is picked **per file, per split**: a file's own `Claimnumber_CLAIM` if it has one;
else that split's features-derived bridge; else (a features file with no own key either)
`raw_v3`'s bridge as the last resort. Every path drops the old `claim_id`, renames
`Claimnumber_CLAIM` → `claim_id`, and writes back under the **original filenames** at their
normal `config.path(...)` locations — so nothing downstream has to change.

**`detection/shap/v3/*.parquet` carries `claim_id` too** (same problem, same fix) — also moved
aside by hand, to `detection/shap/v3/v3_original/` (parquet only; the `_meta.json` sidecars were
left in place and are never touched — nothing in them names the id column).
Kernel: the analysis `.venv` (`python3`).

In [ ]:
# §0 — setup
import shutil
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import config

OLD, NEW = "claim_id", "Claimnumber_CLAIM"   # OLD is what every v3 file currently calls its key


def norm_id(s: pd.Series) -> pd.Series:
    """Dtype, not value, is usually what breaks a claim_id merge: train/test/oot ARE split
    straight out of the same raw data as raw_v3.parquet (confirmed), so their ID_CLAIM values
    should already be a subset of raw's -- an unmatched merge more likely means one side is
    int64 and the other float64 (any NaN upstream turns a whole column to float) or object/
    string, not that the values genuinely differ. Float goes through Int64 first so it prints
    "12345" not "12345.0", then everything becomes a stripped string for comparison.
    """
    if pd.api.types.is_float_dtype(s):
        s = s.astype("Int64")
    return s.astype("string").str.strip()

ORIG_INPUTS = ROOT / "src/data/real/inputs/v3_original"
ORIG_DETECTION = ROOT / "src/data/real/detection/v3_original"
SHAP_DIR = config.path("attributions", "v3", "real", split=config.SPLITS["v3"][0]).parent
SHAP_BACKUP = SHAP_DIR / "v3_original"   # where the shap parquet were moved aside by hand (meta.json stayed put)

print("ROOT =", ROOT)

In [ ]:
# §1 — the bridge: raw_v3's own two id columns, nothing else needed
raw = pd.read_parquet(ORIG_INPUTS / "raw_v3.parquet")
assert OLD in raw.columns and NEW in raw.columns, raw.columns.tolist()
assert raw[NEW].notna().all(), "raw: some rows have no Claimnumber_CLAIM"
assert raw[NEW].is_unique, "raw: Claimnumber_CLAIM is not one row per claim"
bridge = raw[[OLD, NEW]]
print(f"bridge: {len(bridge):,} claims  ({OLD} dtype {raw[OLD].dtype}, {NEW} dtype {raw[NEW].dtype})")

In [ ]:
# §2 — join, drop, rename. Prefers the frame's OWN Claimnumber_CLAIM when it has one — that
# column travelled with this exact row, so it is correct by construction and needs no join.
# Only a frame with NEITHER key is a real problem (raised below, with dtype/range diagnostics).
def rekey(df: pd.DataFrame, name: str, bridge: pd.DataFrame) -> pd.DataFrame:
    if NEW in df.columns:
        out = df.drop(columns=[OLD]) if OLD in df.columns else df.copy()
    else:
        assert OLD in df.columns, f"{name}: has neither {OLD!r} nor {NEW!r}"
        n0 = len(df)
        keyed_bridge = bridge.assign(**{OLD: norm_id(bridge[OLD])})   # index for the join only
        new_vals = norm_id(df[OLD]).map(keyed_bridge.set_index(OLD)[NEW])
        n_bad = int(new_vals.isna().sum())
        if n_bad:
            bad = df.loc[new_vals.isna(), OLD]
            raise AssertionError(
                f"{name}: {n_bad:,} / {n0:,} rows ({n_bad / n0:.1%}) had no match in the bridge "
                f"(compared as normalized strings, so this is not a dtype artefact).\n"
                f"  this file's {OLD}: dtype {df[OLD].dtype}  range [{df[OLD].min()}, {df[OLD].max()}]  "
                f"e.g. {df[OLD].head(3).tolist()}\n"
                f"  bridge's    {OLD}: dtype {bridge[OLD].dtype}  range [{bridge[OLD].min()}, {bridge[OLD].max()}]  "
                f"e.g. {bridge[OLD].head(3).tolist()}\n"
                f"  unmatched sample: {bad.head(5).tolist()}")
        out = df.drop(columns=[OLD]).assign(**{NEW: new_vals.values})
    out = out.rename(columns={NEW: OLD})
    out.insert(0, OLD, out.pop(OLD))
    assert out[OLD].is_unique, f"{name}: claim_id not unique after rekey"
    return out

In [ ]:
# §3 — raw_v3.parquet: just swap the two key columns (it's the source of the bridge, not run through rekey)
raw_out = raw.rename(columns={OLD: "ID_CLAIM", NEW: OLD})
raw_out.insert(0, OLD, raw_out.pop(OLD))
raw_out.to_parquet(config.path("raw_dataset", "v3", "real"), index=False)
print("raw_v3.parquet written:", raw_out.shape, "— claim_id is now Claimnumber_CLAIM; ID_CLAIM kept under its own name")

In [ ]:
# §4 — features / targets / scores, per split.
#
# targets/scores never carry their own Claimnumber_CLAIM (01_export_v3 selected only ID+DATE+
# OBSERVED / ID+SCORE for them) — but they came from the SAME transformed frame as that split's
# features file, so features_v3_{split}'s own ID_CLAIM<->Claimnumber_CLAIM pairing (when it has
# one) is a bridge guaranteed to match, unlike raw_v3 (a SEPARATE single-file extraction that may
# not share the same ID_CLAIM numbering at all — see the 231300-unmatched failure this fixes).
SPLIT_BRIDGES = {}
for split in config.SPLITS["v3"]:
    feat_raw = pd.read_parquet(ORIG_INPUTS / f"features_v3_{split}.parquet")
    if NEW in feat_raw.columns:
        split_bridge = feat_raw[[OLD, NEW]].drop_duplicates(OLD)
        print(f"{split}: features file carries its own {NEW} -- using it as this split's bridge "
              f"({len(split_bridge):,} claims)")
    else:
        split_bridge = bridge
        print(f"{split}: features file has no {NEW} of its own -- falling back to raw's bridge")
    SPLIT_BRIDGES[split] = split_bridge

    rekey(feat_raw, f"features_v3_{split}", split_bridge).to_parquet(
        config.split_path("processed_inputs", "v3", split), index=False)

    tgt = pd.read_parquet(ORIG_INPUTS / f"targets_v3_{split}.parquet")
    rekey(tgt, f"targets_v3_{split}", split_bridge).to_parquet(
        config.split_path("targets", "v3", split), index=False)

    sc = pd.read_parquet(ORIG_DETECTION / f"v3_scores_{split}.parquet")
    rekey(sc, f"v3_scores_{split}", split_bridge).to_parquet(
        config.split_path("scores", "v3", split), index=False)

    print(split, "-- features/targets/scores written")

In [ ]:
# §5 — SHAP attributions: parquet already moved aside by hand to shap/v3/v3_original/;
# _meta.json sidecars were left in place and are never touched here. Attributions carry no
# Claimnumber_CLAIM of their own (phi values + claim_id only), so they use that split's bridge
# from §4 -- the same guaranteed-to-match source targets/scores just used, not raw's.
for split in config.SPLITS["v3"]:
    for p in sorted(SHAP_BACKUP.glob(f"v3_attributions_{split}*.parquet")):
        out = rekey(pd.read_parquet(p), p.name, SPLIT_BRIDGES[split])
        out.to_parquet(SHAP_DIR / p.name, index=False)
        print(p.name, "rekeyed:", out.shape)

In [ ]:
# §6 — sanity: every split's claim_id sets line up across features/targets/scores now
for split in config.SPLITS["v3"]:
    f = pd.read_parquet(config.split_path("processed_inputs", "v3", split))
    t = pd.read_parquet(config.split_path("targets", "v3", split))
    s = pd.read_parquet(config.split_path("scores", "v3", split))
    print(f"{split:<6} features {f['claim_id'].isin(t['claim_id']).mean():.1%} in targets  "
          f"scores {s['claim_id'].isin(t['claim_id']).mean():.1%} in targets  "
          f"sample {f['claim_id'].head(3).tolist()}")

Done. `inputs/v3_original/`, `detection/v3_original/` and `detection/shap/v3/v3_original/` are safe
to delete once `03_01_corrector_inputs.ipynb` and `02_error_inheritance.ipynb`'s v3 cell show
real coverage — keep them until then.